In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def import_data(location: str) -> pd.DataFrame:
    
    if location.endswith('.json'):
        df = pd.read_json(location, lines = True)  
    elif location.endswith('.csv'):
        df = pd.read_csv(location)
    else:
        raise ValueError('Unusable file type')  
    return df

In [ ]:
vet_business = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_business.csv')
vet_check_in = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_check_in.csv')
vet_reviews  = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_reviews.csv')
vet_tip      = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_tip.csv')
vet_user     = import_data('/kaggle/input/datasets/micahluftig/yelp-veterinary-data/yelp_veterinary_user.csv')

In [ ]:
# 1. Initialize the vectorizer
tfidf = TfidfVectorizer(
    max_features = 5000,      
    min_df = 3,                
    max_df = 0.5,               
    ngram_range = (1, 2)       
)

tfidf_matrix = tfidf.fit_transform(vet_reviews['clean_text_tfidf'])

print(tfidf_matrix.shape)

In [ ]:
feature_names = tfidf.get_feature_names_out()

avg_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_n = 30
top_indices = avg_scores.argsort()[-top_n:][::-1]

for i in top_indices:
    print(feature_names[i], round(avg_scores[i], 4))

In [ ]:
low = tfidf.transform(vet_reviews.loc[vet_reviews['stars'] <= 2, 'clean_text'])
high = tfidf.transform(vet_reviews.loc[vet_reviews['stars'] >= 4, 'clean_text'])

low_avg = np.asarray(low.mean(axis = 0)).flatten()
high_avg = np.asarray(high.mean(axis = 0)).flatten()
diff = low_avg - high_avg  

In [ ]:
top_negative = diff.argsort()[-20:][::-1]
print("Words associated with LOW ratings:")
for i in top_negative:
    print(feature_names[i], round(diff[i], 4))

In [ ]:
top_positive = diff.argsort()[:20]  
print("Words associated with HIGH ratings:")
for i in top_positive:
    print(feature_names[i], round(diff[i], 4))

In [ ]:
top_n = 20
top_indices = np.concatenate([top_negative, top_positive])  

results = pd.DataFrame({
    'word': [feature_names[i] for i in top_indices],
    'diff': [diff[i] for i in top_indices]
})
results = results.sort_values('diff')  
results.to_csv('tfidf_diff_top_words.csv', index = False)